In [53]:
import jsonlines
import json
import os
import openai
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-german-cased")
import tiktoken

def load_data(legislatur):
    #legislatur = 20
    alleReden = []
    with jsonlines.open(f'../../data/speeches_{legislatur}.jsonl') as f:
        for line in f.iter():
            #for line in list(f):
            alleReden.append(line)
    alleReden.sort(key = lambda x:x['date'])
    return alleReden

def search_data(reden,keyword):
    #legislatur = 20
    speeches_with_keyword = [r for r in reden if keyword in r['text']]
    return speeches_with_keyword

def parse_json_safely(response: str) -> dict:
    """Tries to parse JSON, strips junk if necessary."""
    try:
        return json.loads(response)
    except json.JSONDecodeError:
        try:
            start, end = response.find("{"), response.rfind("}") + 1
            return json.loads(response[start:end])
        except Exception:
            return {"error": "Invalid JSON", "raw_response": response}

def prompt_associations(rede: str) -> str:
    SCHEMA = {
        "language": "de",
        "associations": [
            {
                "source_concept": "<string>",
                "target_concept": "<string>",
                "source_valence":"<float in [-1.0, 1.0]>",
                "target_valence":"<float in [-1.0, 1.0]>",
                "source_target_logical_coherence": "consistent|inconsistent|neutral",
                "association_type": "claim|data|warrant|backing|qualifier|rebuttal|",
                "association_valence":"<float in [-1.0, 1.0]>",
                "epistemic_frame": "belief|knowledge|uncertainty|inquiry",
                "explicitness_level": "explicit|implicit|fragmentary",
                "evaluation_terms": ["<Wertungen/Adjektive>"],
                "certainty_score": "<float in [0.0, 1.0]>",
                "evidence": ["<exakte Zitatphrase>", "..."],
                "justification": "string"
            }
        ]
    }

    sys_msg = f"""Du bist Expert:in für Diskursanalyse und Argument-Mining und Teil einer Computational Social Science Pipeline für Datenanalyse."""

    N = 8
    user_msg = f"""
    Analysiere die untenstehende REDE. Ziel: Extrahiere die {N-2} - {N+2} wichtigsten Assoziationen und semantischen Verknüpfungen im Text.
    
    Gib AUSSCHLIESSLICH ein JSON-Objekt zurück, das diesem SCHEMA strikt entspricht:
    
    {SCHEMA}
    
    INSTRUCTIONS:
    
    - Verwende ausschließlich die REDE.
    - Wähle sinntragende Konzepte (keine Funktionswörter), ggf. Konzepte zusammenfassen (Singular, konsistente Benennung).
    - Likert und float Scores müssen Wertebereiche einhalten (certainty_score ∈ [0..1], valence ∈ [-1.0..1.0]).
    - Verwende wenn möglich nur exakte Zitate aus der REDE.
    - Sei konsistent, knapp, logisch.
    - Gib keine Erklärungen oder Hinweise außerhalb des SCHEMAs!
    - Nur das SCHEMA zur weiteren Analyse.
    
    ---
    
    REDE:
    \"\"\"{rede["text"]}\"\"\"
    
    """

    messages=[
        {
            "role": "system",
            "content": sys_msg
        },
        {
            "role": "user",
            "content": user_msg
        }
    ]
    return messages

def get_coherence(a: dict) -> str:
    return a.get("source_target_logical_coherence") or "unknown"

ApiKey = os.environ['OPENAI_API_KEY']


In [63]:
keyword = "Digital Services Act"
#keyword = "Social Media"
#keyword = "Impfpflicht"
#keyword = "soziale Medien"
keyword="Lieferkettengesetz"

WP = 20

#model_name = "gpt-4.1-nano-2025-04-14"
#model_name = "gpt-5-mini-2025-08-07"
model_name = "gpt-4.1-mini-2025-04-14"

alleReden = load_data(WP)
selection = search_data(alleReden,keyword)

print(f"There are {len(selection)} Reden with {keyword}")


There are 178 Reden with Lieferkettengesetz


In [59]:
client = openai.OpenAI(api_key=ApiKey)  # dein API-Key
concept_annotations = {}

for sx,speech in enumerate(selection):
    messages = prompt_associations(speech)

    enc = tiktoken.get_encoding("cl100k_base")  # GPT-4/3.5 kompatibel
    tokensSys = enc.encode(messages[0]['content'])
    tokensUse = enc.encode(messages[1]['content'])
    
    #tokensSys = tokenizer.encode(messages[0]['content'])
    #tokensUse = tokenizer.encode(messages[1]['content'])

    print(f"System prompt: {len(messages[0]['content'])} ({len(tokensSys)} tokens)")
    print(f"User prompt: {len(messages[1]['content'])} ({len(tokensUse)} tokens)")

    try:
        response = client.chat.completions.create(
            #model="gpt-3.5-turbo",  # oder "gpt-3.5-turbo"
            model=model_name,  # oder "gpt-3.5-turbo"
            messages=messages
        )   
        raw_response = response.choices[0].message.content
        result = parse_json_safely(raw_response)
        print(f"Speech processed ({sx+1})")
        assocs = result.get("associations", [])
        concept_annotations.update(
            {
                speech['id']:assocs
            }
        )
    except Exception as e:
        print(f"Rede {speech['id']} Error (outer): {e}")
        continue       
        

    

System prompt: 127 (26 tokens)
User prompt: 5249 (1468 tokens)
Speech processed (1)
System prompt: 127 (26 tokens)
User prompt: 6575 (1825 tokens)
Speech processed (2)
System prompt: 127 (26 tokens)
User prompt: 5515 (1514 tokens)
Speech processed (3)
System prompt: 127 (26 tokens)
User prompt: 5684 (1575 tokens)
Speech processed (4)
System prompt: 127 (26 tokens)
User prompt: 11193 (3072 tokens)
Speech processed (5)
System prompt: 127 (26 tokens)
User prompt: 4967 (1390 tokens)
Speech processed (6)
System prompt: 127 (26 tokens)
User prompt: 7476 (2122 tokens)
Speech processed (7)
System prompt: 127 (26 tokens)
User prompt: 6282 (1716 tokens)
Speech processed (8)
System prompt: 127 (26 tokens)
User prompt: 4862 (1374 tokens)
Speech processed (9)
System prompt: 127 (26 tokens)
User prompt: 5433 (1465 tokens)
Speech processed (10)
System prompt: 127 (26 tokens)
User prompt: 6039 (1611 tokens)
Speech processed (11)
System prompt: 127 (26 tokens)
User prompt: 4326 (1244 tokens)
Speech pro

In [60]:
len(concept_annotations)

67

In [61]:
with open(f"annotations/concept_annotations_{keyword} (WP{WP}) ({model_name}).json", "w", encoding="utf-8") as f:
    json.dump(concept_annotations, f, ensure_ascii=False, indent=2)